# Lezione 9. Pianificazione e modelli: quando l'agente impara come funziona il mondo

**DIH DATalife — Corso di Reinforcement Learning (RL) da zero**

Benvenuta o benvenuto alla lezione nove. Oggi fai un salto rispetto a tutto ciò che hai visto finora: invece di un agente che reagisce soltanto all'esperienza, costruisci uno che **impara un modello interno dell'ambiente** e lo usa per pianificare.

## Scaletta della lezione

1. RL senza modello contro RL basato su modello
2. Dyna-Q: imparare agendo e imparare immaginando
3. MCTS, in breve, come ponte verso il capitolo 12
4. Codice dal vivo: Dyna-Q contro Q-learning puro in un mondo a griglia con muri
5. Punti chiave e cosa ti aspetta nella lezione 10


## 1. RL senza modello contro RL basato su modello

*Traccia della lezione: parti dall'analogia della mappa. Guidare alla cieca correggendo ogni svolta è RL senza modello; aprire una mappa mentale e pianificare il percorso prima di partire è RL basato su modello.*

Tutto ciò che hai visto fino alla lezione otto —bandit, Q-learning, gradienti di politica, actor-critic— appartiene alla famiglia dell'RL **senza modello**: l'agente impara direttamente una politica o una funzione di valore dall'esperienza, senza rappresentare in modo esplicito come cambia l'ambiente da un passo all'altro.

L'RL **basato su modello** aggiunge un elemento: un modello delle dinamiche dell'ambiente, formato da una funzione di transizione (quale stato viene dopo) e da una funzione di ricompensa (quale ricompensa si ottiene). Questo modello può essere disponibile in anticipo o, cosa più comune, essere appreso dall'esperienza.

Pensa a un modello come a un'approssimazione di un Processo Decisionale di Markov (MDP): non deve essere una replica esatta dell'ambiente, basta che sia abbastanza fedele da orientare le decisioni. Esistono tre tipi di modello, e ciascuno fallisce in modo diverso:

- **Modello di aspettativa**: prevede lo stato successivo e la ricompensa attesi, in media. Quella media può non corrispondere a nessuno stato reale (un robot che "allucina" un buco in un muro che non esiste)
- **Modello stocastico o generativo**: genera campioni di stati e ricompense validi, con nuovo rumore a ogni interrogazione. Non restituisce mai uno stato impossibile, ma introduce più varianza
- **Modello di dinamica completa**: apprende l'intera distribuzione di probabilità sugli stati successivi. Negli spazi continui richiede di lavorare con integrali, cosa che di solito si approssima con metodi come Markov Chain Monte Carlo (MCMC)

Un modello può essere rappresentato come tabella, come funzione lineare, come rete neurale o anche in modo non parametrico: un buffer di replay dell'esperienza, dove "interrogare il modello" significa estrarre a caso una transizione reale che hai già osservato. Quest'ultimo caso è proprio quello che programmerai oggi.


## 2. Dyna-Q: imparare agendo e imparare immaginando

*Traccia della lezione: disegna alla lavagna le due fasi del ciclo prima di toccare il codice. Insisti sul fatto che Dyna-Q con zero passi di pianificazione È Q-learning puro, non un'approssimazione.*

La famiglia Dyna, proposta da Richard Sutton, combina due fonti di apprendimento che finora avevi trattato separatamente: l'esperienza reale e l'esperienza simulata da un modello appreso.

Dyna-Q, la sua versione tabulare, funziona in due fasi all'interno di ogni passo:

1. **Apprendimento diretto**: l'agente agisce nell'ambiente reale, osserva lo stato successivo e la ricompensa, applica un normale aggiornamento di Q-learning e salva quella transizione nel suo modello interno
2. **Pianificazione simulata**: dopo ogni passo reale, l'agente ripete *n* volte quanto segue: sceglie a caso una combinazione di stato e azione già osservata in precedenza, chiede al modello cosa accadrebbe e applica un altro aggiornamento di Q-learning su quella transizione immaginata

Quel numero *n* si chiama **passi di pianificazione**. Con *n* uguale a zero, Dyna-Q si riduce esattamente a Q-learning puro. Più passi di pianificazione aggiungi, più l'agente "ripassa" la propria esperienza prima di tornare ad agire, e meno episodi reali gli servono per convergere, al prezzo di più calcolo per ogni passo reale.

**Una precisazione importante quando l'ambiente cambia:**

- Se un percorso che prima era libero si blocca, Dyna-Q aggiorna il suo modello appena urta l'ostacolo nuovo e subito ripianifica
- Se compare una scorciatoia nuova che l'agente non ha mai provato, Dyna-Q può impiegare molto a scoprirla, perché nulla gli indica che esista finché l'esplorazione non lo porta lì per caso
- **Dyna-Q+** aggiunge un bonus di esplorazione che cresce quanto più a lungo non si visita una combinazione di stato e azione, il che aiuta a trovare più in fretta le scorciatoie nuove

Un modello appreso non è mai perfetto. Combinare la pianificazione basata su modello con aggiornamenti senza modello —ricadendo su questi ultimi quando il modello risulta poco affidabile— ti dà il meglio di entrambi i mondi: l'orientamento strutturato di un modello e la robustezza di imparare dalla realtà.


## 3. MCTS: esplorare un albero di futuri possibili

*Traccia della lezione: menziona MCTS come preparazione ad AlphaZero nella lezione 12, senza approfondire il codice oggi.*

La ricerca di Monte Carlo su alberi (Monte Carlo Tree Search, MCTS) porta la pianificazione un passo più in là: invece di simulare singole transizioni come Dyna-Q, costruisce e mantiene un albero di futuri possibili a partire dallo stato attuale, crescendo solo nella parte che effettivamente esplora.

Ogni ricerca si organizza in quattro fasi che si ripetono quante volte il budget di calcolo lo consente:

1. **Selezione**: dalla radice, percorri i nodi già esistenti con una politica di albero, fino a un nodo che non è ancora stato espanso del tutto
2. **Espansione**: aggiungi un nuovo nodo figlio, corrispondente a un'azione non ancora provata da lì
3. **Simulazione (rollout)**: a partire da quel nodo nuovo, gioca una traiettoria completa con una politica rapida, spesso casuale, fino a uno stato terminale
4. **Retropropagazione**: propaga il risultato verso l'alto, aggiornando il valore e il contatore di visite di ogni nodo del percorso

Il valore di un nodo è la media dei risultati di tutte le simulazioni che vi sono passate. Upper Confidence Bound for Trees (UCT) decide quale ramo esplorare con la stessa logica che Upper Confidence Bound (UCB) applicava ai bandit della lezione tre: somma al valore stimato un bonus di esplorazione che cresce quando il nodo è stato visitato poco.

MCTS con UCT è il motore di ricerca dietro AlphaGo. Due tecniche affini, il Controllo Predittivo basato su Modello (MPC) e MuZero, sono fuori dalla portata di questa lezione.


## 4. Codice dal vivo: Dyna-Q contro Q-learning puro

*Traccia della lezione: costruisci l'esempio cella per cella. Chiedi alla classe di prevedere, prima di eseguire il grafico finale, quale curva scenderà più in fretta.*

Costruirai un piccolo mondo a griglia con due muri che obbligano l'agente a girarci intorno per raggiungere l'obiettivo. Confronterai il Q-learning puro (zero passi di pianificazione) con due versioni di Dyna-Q (cinque e 50 passi di pianificazione).

Ti servono solo `numpy` per il calcolo e `matplotlib` per il grafico: entrambi sono preinstallati in Google Colab.

**Inizia importando le librerie e fissando il seme, perché l'esperimento sia riproducibile.**


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)


### Il mondo a griglia

Definiamo una griglia di sei righe per nove colonne. L'agente parte dall'angolo in basso a sinistra e l'obiettivo si trova nell'angolo in alto a destra. Due segmenti di muro, `PAREDES`, obbligano a girare intorno invece di andare in linea retta.

Le quattro azioni possibili sono su, giù, sinistra e destra.


In [ ]:
# --- Definimos un mundo-cuadrícula pequeño, hecho a mano ---
ALTO, ANCHO = 6, 9                 # 6 filas, 9 columnas
INICIO = (5, 0)                    # esquina inferior izquierda
META = (0, 8)                      # esquina superior derecha

# Dos segmentos de muro que obligan a rodear en vez de ir en línea recta
PAREDES = {(1, 3), (2, 3), (3, 3), (0, 6), (1, 6), (2, 6)}

# Cuatro acciones posibles: arriba, abajo, izquierda, derecha
ACCIONES = [(-1, 0), (1, 0), (0, -1), (0, 1)]
N_ACCIONES = len(ACCIONES)


### La dinamica dell'ambiente

`paso_entorno` applica un'azione in modo deterministico: se il movimento urta un muro o il bordo del mondo, l'agente resta dov'era. Ogni passo dà una ricompensa di -1, tranne l'arrivo all'obiettivo, per incentivare il percorso più breve.


In [ ]:
def paso_entorno(estado, accion_idx):
    """Aplica una acción de forma determinista: si el movimiento choca contra
    un muro o el borde del mundo, el agente se queda donde estaba."""
    if estado == META:
        return estado, 0.0, True
    d_fila, d_col = ACCIONES[accion_idx]
    fila, col = estado
    nueva_fila = min(max(fila + d_fila, 0), ALTO - 1)
    nueva_col = min(max(col + d_col, 0), ANCHO - 1)
    nuevo_estado = (nueva_fila, nueva_col)
    if nuevo_estado in PAREDES:
        nuevo_estado = estado
    terminado = nuevo_estado == META
    recompensa = 0.0 if terminado else -1.0   # -1 por paso incentiva el camino corto
    return nuevo_estado, recompensa, terminado


### Tabella Q e politica epsilon-greedy

`indice` converte una casella (riga, colonna) in un indice unico di riga nella tabella Q. `elegir_accion` usa la stessa politica epsilon-greedy che conosci già dalla lezione tre.


In [ ]:
def indice(estado):
    """Convierte una casilla (fila, columna) en un índice único de fila en la tabla Q."""
    return estado[0] * ANCHO + estado[1]


def elegir_accion(Q, estado, epsilon):
    """Política epsilon-greedy, la misma que usaste en la clase tres."""
    if np.random.random() < epsilon:
        return np.random.randint(N_ACCIONES)
    fila_q = Q[indice(estado)]
    mejores = np.flatnonzero(fila_q == np.max(fila_q))
    return np.random.choice(mejores)


### L'agente Dyna-Q

Qui sta il cuore della lezione. Osserva le due fasi dentro il ciclo `while`:

- **Fase 1** (apprendimento diretto): agisce, osserva, aggiorna Q con l'esperienza reale e salva la transizione in `modelo`
- **Fase 2** (pianificazione simulata): ripete `n_planning` volte l'operazione di campionare una transizione già vissuta e aggiornare Q con essa, senza toccare di nuovo l'ambiente reale

Con `n_planning=0` non viene eseguita nessuna iterazione della fase 2: l'agente si riduce a Q-learning puro.


In [ ]:
def ejecutar_dyna_q(n_planning, n_episodios, alpha=0.1, gamma=0.95, epsilon=0.1, max_pasos=1000):
    """Entrena un agente Dyna-Q tabular. Con n_planning=0 es Q-learning puro:
    solo queda la actualización con experiencia real, sin repetir nada."""
    Q = np.zeros((ALTO * ANCHO, N_ACCIONES))
    modelo = {}                      # modelo aprendido: (estado, accion) -> (estado_siguiente, recompensa)
    pasos_por_episodio = np.zeros(n_episodios)

    for ep in range(n_episodios):
        estado = INICIO
        pasos = 0
        terminado = False

        while not terminado and pasos < max_pasos:
            # --- Fase 1: aprendizaje directo, con experiencia real ---
            accion = elegir_accion(Q, estado, epsilon)
            sig_estado, recompensa, terminado = paso_entorno(estado, accion)

            i_estado, i_sig = indice(estado), indice(sig_estado)
            objetivo = recompensa + gamma * np.max(Q[i_sig])
            Q[i_estado, accion] += alpha * (objetivo - Q[i_estado, accion])

            modelo[(estado, accion)] = (sig_estado, recompensa)

            # --- Fase 2: planificación simulada, con experiencia imaginada ---
            transiciones_vistas = list(modelo.items())
            for _ in range(n_planning):
                (s_sim, a_sim), (s_sig_sim, r_sim) = transiciones_vistas[
                    np.random.randint(len(transiciones_vistas))
                ]
                i_sim, i_sig_sim = indice(s_sim), indice(s_sig_sim)
                objetivo_sim = r_sim + gamma * np.max(Q[i_sig_sim])
                Q[i_sim, a_sim] += alpha * (objetivo_sim - Q[i_sim, a_sim])

            estado = sig_estado
            pasos += 1

        pasos_por_episodio[ep] = pasos

    return pasos_por_episodio


### Confrontiamo tre configurazioni

Addestriamo con zero, cinque e 50 passi di pianificazione, e facciamo la media di dieci ripetizioni di ciascuna per attenuare il caso nell'esplorazione.

**Esegui la cella e osserva quanto tempo impiega ciascuna configurazione: anche il costo di calcolo extra fa parte del confronto.**


In [ ]:
# --- Comparamos tres configuraciones: 0, 5 y 50 pasos de planificación ---
n_episodios = 50
n_repeticiones = 10       # promediamos varias corridas para suavizar el azar
configuraciones = [0, 5, 50]
resultados = {}

for n_planning in configuraciones:
    acumulado = np.zeros(n_episodios)
    for _ in range(n_repeticiones):
        acumulado += ejecutar_dyna_q(n_planning, n_episodios)
    resultados[n_planning] = acumulado / n_repeticiones

print("Entrenamiento terminado para", configuraciones, "pasos de planificación")


### Disegniamo il confronto

**Esegui questa cella per vedere le tre curve.**


In [ ]:
# --- Graficamos la comparación ---
plt.figure(figsize=(10, 5))
for n_planning in configuraciones:
    etiqueta = "Q-learning puro (0 pasos)" if n_planning == 0 else f"Dyna-Q ({n_planning} pasos de planificación)"
    plt.plot(resultados[n_planning], label=etiqueta)
plt.xlabel("Episodio")
plt.ylabel("Pasos hasta llegar a la meta")
plt.title("Dyna-Q vs. Q-learning puro en un mundo-cuadrícula con muros")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


**Lettura del risultato**

*Traccia della lezione: fermati su questo grafico e chiedi alla classe quale curva si aspettava e quale vede.*

Le tre curve scendono, ma a ritmi molto diversi. Quella con zero passi di pianificazione (Q-learning puro) migliora lentamente, perché impara solo da ciò che vive in tempo reale. Le curve con cinque e, soprattutto, 50 passi di pianificazione scendono molto più in fretta: ogni passo reale viene sfruttato molte volte in più, grazie agli aggiornamenti simulati sul modello che l'agente stesso ha costruito.


## Punti chiave

- L'RL basato su modello aggiunge una funzione di transizione e una funzione di ricompensa che l'agente usa per pianificare, invece di reagire soltanto all'esperienza diretta
- Esistono tre tipi di modello con modalità di errore diverse: il modello di aspettativa può prevedere stati che non esistono, il modello stocastico genera sempre stati validi al prezzo di più varianza, e il modello di dinamica completa cattura tutta la distribuzione, a maggiore costo computazionale
- Dyna-Q combina, a ogni passo, un aggiornamento con esperienza reale e vari aggiornamenti con esperienza simulata dal modello appreso; con zero passi di pianificazione, si riduce a Q-learning puro
- Dyna-Q+ aggiunge un bonus di esplorazione che cresce nel tempo dall'ultima visita, il che aiuta a scoprire scorciatoie nuove che Dyna-Q impiegherebbe a trovare
- MCTS esplora un albero di futuri possibili in quattro fasi (selezione, espansione, simulazione e retropropagazione), e UCT bilancia esplorazione e sfruttamento con la stessa idea di UCB nei bandit della lezione tre
- Un modello appreso non è mai perfetto: combinarlo con aggiornamenti senza modello dà robustezza quando il modello sbaglia

## Prima della prossima lezione

**Apri il notebook di pratica di questa lezione e completa i due esercizi su Dyna-Q.** Programmerai tu stesso il passo di pianificazione simulata da zero e misurerai l'effetto reale di variare i passi di pianificazione.

Nella lezione 10 torni all'RL profondo senza modello, ma con le tecniche più avanzate del campo: Rainbow DQN e l'RL distribuzionale.
